# KREG 数据分析

这份 notebook 面向 `minimal_adl_ethene_butadiene_KREG` 的新流程，核心关注点不再是静态 pool 排名，而是：

- TS seed 是否解析正确
- 累计已标注样本是否持续增长
- KREG 训练结果是否可用
- round 1 的 MD 轨迹帧与选帧结果是否合理


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


In [ ]:
def find_experiment_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        direct = base / 'minimal_adl_ethene_butadiene_KREG'
        if (direct / 'configs' / 'base.yaml').exists():
            return direct
        if (base / 'configs' / 'base.yaml').exists() and (base / 'scripts').exists():
            return base
    raise FileNotFoundError('Could not locate minimal_adl_ethene_butadiene_KREG from the current notebook cwd.')

EXPERIMENT_ROOT = find_experiment_root()
EXPERIMENT_ROOT


In [ ]:
def safe_read_json(path: Path, default):
    if not path.exists():
        return default
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

paths = {
    'ts_seed_summary': EXPERIMENT_ROOT / 'results' / 'ts_seed_summary.json',
    'cumulative_manifest': EXPERIMENT_ROOT / 'data' / 'processed' / 'cumulative_labeled_manifest.json',
    'dataset_metadata': EXPERIMENT_ROOT / 'data' / 'processed' / 'delta_dataset_metadata.json',
    'training_summary': EXPERIMENT_ROOT / 'models' / 'training_summary.json',
    'md_frame_manifest': EXPERIMENT_ROOT / 'results' / 'round_001_md_frame_manifest.json',
    'selection_summary': EXPERIMENT_ROOT / 'results' / 'round_001_selection_summary.json',
}

ts_seed_summary = safe_read_json(paths['ts_seed_summary'], {})
cumulative_manifest = safe_read_json(paths['cumulative_manifest'], {'samples': []})
dataset_metadata = safe_read_json(paths['dataset_metadata'], {'samples': []})
training_summary = safe_read_json(paths['training_summary'], {})
md_frame_manifest = safe_read_json(paths['md_frame_manifest'], {'frames': []})
selection_summary = safe_read_json(paths['selection_summary'], {})

overview = {
    'experiment_root': str(EXPERIMENT_ROOT),
    'ts_num_atoms': ts_seed_summary.get('num_atoms'),
    'ts_num_imaginary_frequencies': ts_seed_summary.get('num_imaginary_frequencies'),
    'ts_first_imaginary_frequency': ts_seed_summary.get('first_imaginary_frequency'),
    'cumulative_labeled_samples': len(cumulative_manifest.get('samples', [])),
    'dataset_num_samples': dataset_metadata.get('num_samples'),
    'md_num_frames_round_001': md_frame_manifest.get('num_frames'),
    'round_001_selected_count': selection_summary.get('selected_count'),
}
overview


In [ ]:
selected_df = pd.DataFrame(dataset_metadata.get('samples', []))
frames_df = pd.DataFrame(md_frame_manifest.get('frames', []))
selection_df = pd.DataFrame([{
    'round_index': selection_summary.get('round_index'),
    'selected_count': selection_summary.get('selected_count'),
    'num_uncertain_samples': selection_summary.get('num_uncertain_samples'),
    'uncertain_ratio': selection_summary.get('uncertain_ratio'),
    'converged': selection_summary.get('converged'),
}])

display(selection_df)
display(selected_df.head())
display(frames_df[['sample_id', 'trajectory_id', 'time_fs', 'uncertainty']].head()) if not frames_df.empty else None
